In [15]:
# Let's start with dependency installations
# pip install langchain_openai langchain_core langgraph langgraph-supervisor langchain
# pip install --upgrade langgraph langgraph-supervisor

# Import Key first
import os, getpass
import logging

def _set_env(var: str):
    logging.warning("Calling setEnv")
    if not os.environ.get(var):
        logging.warning("API Key not found. Let's input")
        os.environ[var] = getpass.getpass(f"{var}: ")
    else:
        logging.warning("Found API Key in the environment")

_set_env("OPENAI_API_KEY")


In [16]:
import datetime
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm_with_no_tools = ChatOpenAI(model="gpt-4o")
print("Done with creation of an LLM")

def currentdatetime() -> str:
    "This tool provide the current date, time and day of the week in your location."
    dt = datetime.datetime.now()
    return "Current date is {} and today is {}".format(dt, dt.strftime("%A"))

# https://reference.langchain.com/python/langchain/agents/factory/create_agent
time_keeper_agent = create_agent(
    model = llm_with_no_tools,
    tools = [currentdatetime],
    name = "time_keeper_agent",
    system_prompt = "You tell the current date, time and day of the week in your location."
)

Done with creation of an LLM


In [17]:
def shoopeninghours() -> str:
    "This tool provide the daily shop's operating hours."
    schedule = """Monday 10:00am to 08:00pm
                  Tuesday 10:00am to 08:00pm
                  Wednesday 10:00am to 08:00pm
                  Thursday 10:00am to 08:00pm
                  Friday 10:00am to 01:00pm
                  Saturday 11:00am to 05:00pm
                  Sunday 01:00pm to 06:00pm
                """
    
    return schedule;

# https://reference.langchain.com/python/langchain/agents/factory/create_agent
schedule_keeper_agent = create_agent(
    model = llm_with_no_tools,
    tools = [shoopeninghours],
    name = "schedule_keeper_agent",
    system_prompt = "You tell the shop's operating hours for each day of the week."
)

In [18]:
from langgraph_supervisor import create_supervisor

prompt = """You are team supervisor to manage time_keeper_agent and schedule_keeper_agent. 
            Your job is to respond to shop opening hours queries using time_keeper_agent and schedule_keeper_agent.
            To get the current time and date, use time_keeper_agent and for getting shop's operating hours
            use schedule_keeper_agent. You can use both agents in parallel to get require information.
            """

# Remember supervisor controls all communication, task routing and agent selection based on the query context
sv = create_supervisor(
    model = llm_with_no_tools,
    agents = [schedule_keeper_agent, time_keeper_agent],
    prompt = prompt,
    output_mode = "full_history",
    parallel_tools_calls = True
)

In [ ]:
app = sv.compile()

result = app.invoke({
    "messages" : [
        {
            "role": "user", 
            #"content": "What are the shop's opening hours on Monday? Is the shop open right now?"
            "content": "On the basis of current date and time, tell me if the shop open right now?"
         }
    ]
})

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

On the basis of current date and time, tell me if the shop open right now?
================================== Ai Message ==================================
Name: supervisor
Tool Calls:
  transfer_to_time_keeper_agent (call_1GGuxAzS1OX3EITYRAGQdJSZ)
 Call ID: call_1GGuxAzS1OX3EITYRAGQdJSZ
  Args:
================================= Tool Message =================================
Name: transfer_to_time_keeper_agent

Successfully transferred to time_keeper_agent
================================== Ai Message ==================================
Name: time_keeper_agent

I am unable to determine the shop's opening hours without specific information. Could you please provide the opening and closing times of the shop and its location or time zone?
================================== Ai Message ==================================
Name: time_keeper_agent

Transferring back to supervisor
Tool Calls:
  transfer_back_to_supe